# Multi-Seed Variance Testing
Repeating each already-tuned model's final fit across multiple random seeds tests whether a "best model" claim is robust to seed-to-seed stochasticity (this notebook), followed by paired significance testing on the resulting winner (notebook `08`).

Tests whether each target/dataset combination's "best model" claim (established in notebooks 05/05b/05c on a single seed) survives when the same already-tuned `best_params` are refit at several different random seeds. This is a deployment-stochasticity check (RandomForest bootstrap + per-split feature-subset sampling; XGBoost/LightGBM row/column subsampling), not a re-tuning pass -- no Optuna search happens in this notebook, and none of the six canonical, already-documented target/dataset results are modified or re-run by anything here.

**Scope**: 5 seeds per model generally, 3 for F-DATA RandomForest specifically (its own final-fit cost, ~20-30 min/fit, justified the smaller count). Results consumed by notebook `08` for paired significance testing once all six combinations are done here.

Every cell below loads already-computed results from `data/interim/*.joblib` rather than recomputing them -- the actual sweeps were run as standalone scripts via `systemd-run --user --scope` (fresh process per model, same discipline as notebooks 05c), documented in full in this notebook's own README companion, `scripts/decision6_seed_variance/README.md`. This notebook's job is to load, tabulate, and report those results as a real, executed artifact, not to re-run the (in some cases hours-long) fits inline.

**Order**: F-DATA power and F-DATA duration were prioritized first (tightest single-seed model gaps, most likely to have a best-model claim overturned by noise). The remaining four follow in their original completion order: F-DATA memory, PM100 power, PM100 memory, PM100 duration.

In [1]:
import sys
sys.path.append("..")
import joblib
import numpy as np
import pandas as pd

INTERIM_DIR = "../data/interim"

def summarize(seed_results):
    r2s = [v["R2"] for v in seed_results.values()]
    return pd.Series({
        "n_seeds": len(r2s),
        "mean": np.mean(r2s), "std": np.std(r2s),
        "min": min(r2s), "max": max(r2s),
    })


def pairwise_overlaps(summary_df):
    """Real min/max overlap check between every pair of models/variants in a summary
    table (as built by `summarize`) -- computed, not eyeballed. Returns a DataFrame of
    booleans, True where the two ranges genuinely overlap (any shared region, however
    small counts as an overlap; there is no 'razor-thin, ignore it' exception)."""
    names = list(summary_df.index)
    rows = []
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = summary_df.loc[names[i]], summary_df.loc[names[j]]
            lo, hi = max(a["min"], b["min"]), min(a["max"], b["max"])
            rows.append({"pair": f"{names[i]} vs {names[j]}", "overlap": lo <= hi,
                         "region": f"[{lo:.4f}, {hi:.4f}]" if lo <= hi else "--"})
    return pd.DataFrame(rows).set_index("pair")


## Part 1: F-DATA Power

Prioritized first: the three models' single-seed headline numbers landed within a 0.0033 R² band (RandomForest 0.9856, XGBoost 0.9888, LightGBM 0.9855), the tightest gap of any target/dataset combination in this thesis.

XGBoost needed a two-stage approach. A cheap first pass (5 seeds, 1M-row sample, `tree_method="exact"`) came back with std=0.0042, range 0.0122 wide -- roughly 4x the entire gap separating the three models' headline numbers, a real trigger to escalate rather than noise to accept. The full 16.3M-row refit (5 seeds) came back six times tighter (std=0.0007), confirming the reduced-scale sweep was a legitimate but noisier upper-bound proxy, not the final answer.

In [2]:
rf = joblib.load(f"{INTERIM_DIR}/fdata_power_seedvar_rf.joblib")
lgbm = joblib.load(f"{INTERIM_DIR}/fdata_power_seedvar_lgbm.joblib")
xgb_1m = joblib.load(f"{INTERIM_DIR}/fdata_power_seedvar_xgb.joblib")
xgb_full = joblib.load(f"{INTERIM_DIR}/fdata_power_seedvar_xgb_fullscale.joblib")

print("XGBoost, 1M-row sample sweep (first pass, before escalation):")
print({s: round(v["R2"], 4) for s, v in xgb_1m["seed_results"].items()})
print(summarize(xgb_1m["seed_results"]).round(4).to_string())
print()
print("Escalated to full 16.3M-row exact refit -- range came back tight, so this is final.")


XGBoost, 1M-row sample sweep (first pass, before escalation):
{0: 0.9812, 1: 0.9851, 2: 0.9753, 3: 0.98, 4: 0.9875}
n_seeds    5.0000
mean       0.9818
std        0.0042
min        0.9753
max        0.9875

Escalated to full 16.3M-row exact refit -- range came back tight, so this is final.


In [3]:
power_summary = pd.DataFrame({
    "RandomForest": summarize(rf["seed_results"]),
    "LightGBM": summarize(lgbm["seed_results"]),
    "XGBoost (full-scale)": summarize(xgb_full["seed_results"]),
}).T
print("F-DATA power -- full-scale seed-variance summary, all three models directly comparable:")
display(power_summary.round(4))

print("\nPairwise range overlap (computed, not eyeballed -- any shared region counts):")
display(pairwise_overlaps(power_summary))

best = power_summary["mean"].idxmax()
print(f"\nBest model: {best}. Its range does not overlap with either other model's -- "
      "the best-model claim survives seed variance robustly.")
print("The RandomForest-vs-LightGBM (2nd/3rd place) comparison DOES overlap (see table above) -- "
      "that ordering is not reliably established, only the top spot is.")


F-DATA power -- full-scale seed-variance summary, all three models directly comparable:


,n_seeds,mean,std,min,max
RandomForest,3.0,0.9856,0.0007,0.9847,0.9864
LightGBM,5.0,0.9846,0.0015,0.9817,0.9857
XGBoost (full-scale),5.0,0.9893,0.0007,0.9885,0.9903



Pairwise range overlap (computed, not eyeballed -- any shared region counts):


,overlap,region
pair,,
RandomForest vs LightGBM,True,"[0.9847, 0.9857]"
RandomForest vs XGBoost (full-scale),False,--
LightGBM vs XGBoost (full-scale),False,--



Best model: XGBoost (full-scale). Its range does not overlap with either other model's -- the best-model claim survives seed variance robustly.
The RandomForest-vs-LightGBM (2nd/3rd place) comparison DOES overlap (see table above) -- that ordering is not reliably established, only the top spot is.


## Part 2: F-DATA Duration (Result 2)

Prioritized second: notebook 05's own history flagged this as a comparably tight gap (XGBoost 0.836 vs. LightGBM 0.831 vs. RandomForest 0.821). No saved model objects existed for Result 2 (it lives only as printed numbers in the original notebook, a single in-kernel run reusing Result 1's tuned `best_params` on the extended-rolling-window feature set), so every seed here -- including seed 0 -- was computed fresh from a faithful reconstruction of the exact Result 2 pipeline (same 38-month load, `mszl` fix, window=5/20/50 rolling-stat columns, embedding PCA fit on the same stratified tuning sample, same stored `best_params`).

A hist-vs-exact check for XGBoost (closing out the one asymmetry with how F-DATA power was verified) is included below once available.

In [4]:
rf_d = joblib.load(f"{INTERIM_DIR}/fdata_duration_r2_seedvar_rf.joblib")
xgb_d = joblib.load(f"{INTERIM_DIR}/fdata_duration_r2_seedvar_xgb.joblib")
lgbm_d = joblib.load(f"{INTERIM_DIR}/fdata_duration_r2_seedvar_lgbm.joblib")

duration_summary = pd.DataFrame({
    "RandomForest": summarize(rf_d["seed_results"]),
    "LightGBM": summarize(lgbm_d["seed_results"]),
    "XGBoost": summarize(xgb_d["seed_results"]),
}).T
print("F-DATA duration Result 2 -- seed-variance summary, all three models directly comparable:")
display(duration_summary.round(4))

print("\nPairwise range overlap (computed, not eyeballed):")
display(pairwise_overlaps(duration_summary))

best_d = duration_summary["mean"].idxmax()
print(f"\nBest model: {best_d}. Ranges do not overlap across any of the three models -- "
      "the original Result 2 ranking (XGBoost > LightGBM > RandomForest) survives seed variance.")


F-DATA duration Result 2 -- seed-variance summary, all three models directly comparable:


,n_seeds,mean,std,min,max
RandomForest,3.0,0.8213,0.0011,0.8205,0.8228
LightGBM,5.0,0.8266,0.0024,0.8237,0.8306
XGBoost,5.0,0.8376,0.0015,0.8359,0.8401



Pairwise range overlap (computed, not eyeballed):


,overlap,region
pair,,
RandomForest vs LightGBM,False,--
RandomForest vs XGBoost,False,--
LightGBM vs XGBoost,False,--



Best model: XGBoost. Ranges do not overlap across any of the three models -- the original Result 2 ranking (XGBoost > LightGBM > RandomForest) survives seed variance.


In [5]:
import pathlib
_check_path = pathlib.Path(f"{INTERIM_DIR}/fdata_duration_r2_xgb_histexact_check.joblib")
if _check_path.exists():
    check = joblib.load(_check_path)
    print(f"XGBoost hist-vs-exact check (F-DATA duration Result 2, seed=0, full 16.3M-row train split):")
    print(f"  hist:  R2={check['hist_r2']:.4f}")
    print(f"  exact: R2={check['exact_r2']:.4f}  (fit time {check['exact_elapsed']:.1f}s)")
    gap = abs(check["hist_r2"] - check["exact_r2"])
    if gap < 0.01:
        print(f"  Gap ({gap:.4f}) is small -- confirms hist was the right call here, no F-DATA-power-style "
              "bug on this target. Both prioritized combinations now verified on the same standard.")
    else:
        print(f"  Gap ({gap:.4f}) is notable -- worth a closer look before treating hist as settled here.")
else:
    print("Check not yet complete -- see EXPERIMENT_TRACKER.md for the latest status.")


XGBoost hist-vs-exact check (F-DATA duration Result 2, seed=0, full 16.3M-row train split):
  hist:  R2=0.8359
  exact: R2=0.8366  (fit time 5346.5s)
  Gap (0.0007) is small -- confirms hist was the right call here, no F-DATA-power-style bug on this target. Both prioritized combinations now verified on the same standard.


## Part 3: F-DATA Memory

Third combination, in original completion order. XGBoost's `hist`-vs-`exact` check (done proactively this time) found a real 0.0354 gap at seed 0 -- not F-DATA-power-catastrophic, but enough to warrant a full 5-seed `exact` sweep. That sweep did not resolve into a clean answer: it revealed XGBoost itself is seed-unstable on this target under either tree method, including one severe outlier (`exact` seed 4, R²=0.7354) that was investigated directly (diagnostics below) before drawing any conclusion.

In [6]:
rf_m = joblib.load(f"{INTERIM_DIR}/fdata_memory_seedvar_rf.joblib")
xgb_hist_m = joblib.load(f"{INTERIM_DIR}/fdata_memory_seedvar_xgb.joblib")
xgb_exact_m = joblib.load(f"{INTERIM_DIR}/fdata_memory_seedvar_xgb_exact.joblib")
lgbm_m = joblib.load(f"{INTERIM_DIR}/fdata_memory_seedvar_lgbm.joblib")

memory_summary = pd.DataFrame({
    "RandomForest": summarize(rf_m["seed_results"]),
    "XGBoost (hist)": summarize(xgb_hist_m["seed_results"]),
    "XGBoost (exact)": summarize(xgb_exact_m["seed_results"]),
    "LightGBM": summarize(lgbm_m["seed_results"]),
}).T
print("F-DATA memory -- seed-variance summary, all variants:")
display(memory_summary.round(4))

print("\nPairwise range overlap (computed, not eyeballed):")
display(pairwise_overlaps(memory_summary))


F-DATA memory -- seed-variance summary, all variants:


,n_seeds,mean,std,min,max
RandomForest,3.0,0.8210,0.0044,0.8149,0.8246
XGBoost (hist),5.0,0.8358,0.0152,0.8070,0.8498
XGBoost (exact),5.0,0.8203,0.0427,0.7354,0.8495
LightGBM,5.0,0.8591,0.0070,0.8511,0.8681



Pairwise range overlap (computed, not eyeballed):


,overlap,region
pair,,
RandomForest vs XGBoost (hist),True,"[0.8149, 0.8246]"
RandomForest vs XGBoost (exact),True,"[0.8149, 0.8246]"
RandomForest vs LightGBM,False,--
XGBoost (hist) vs XGBoost (exact),True,"[0.8070, 0.8495]"
XGBoost (hist) vs LightGBM,False,--
XGBoost (exact) vs LightGBM,False,--


In [7]:
d2 = joblib.load(f"{INTERIM_DIR}/fdata_memory_xgb_exact_seed2_diagnostic.joblib")
d4 = joblib.load(f"{INTERIM_DIR}/fdata_memory_xgb_exact_seed4_diagnostic.joblib")

print("exact-sweep seed 2 (best, R2=0.8495) vs seed 4 (worst, R2=0.7354) diagnostics:")
diag = pd.DataFrame({
    "seed 2": {"train_R2_log": d2["train_r2"], "test_R2_log": d2["test_r2_log"], "test_R2_raw": d2["test_r2_raw"]},
    "seed 4": {"train_R2_log": d4["train_r2"], "test_R2_log": d4["test_r2_log"], "test_R2_raw": d4["test_r2_raw"]},
}).T
display(diag.round(4))
print("\nTrain performance nearly identical; the gap is concentrated in raw-space (extreme-tail-"
      "dominated) test R2, not log-space -- consistent with a small-ensemble (79 trees) model that "
      "leans on a different, less-generalizable set of secondary embedding features per seed, rather "
      "than a training-divergence or numerical-instability bug. No NaN/inf in either fit.")


exact-sweep seed 2 (best, R2=0.8495) vs seed 4 (worst, R2=0.7354) diagnostics:


,train_R2_log,test_R2_log,test_R2_raw
seed 2,0.9497,0.8831,0.8495
seed 4,0.9463,0.8601,0.7354



Train performance nearly identical; the gap is concentrated in raw-space (extreme-tail-dominated) test R2, not log-space -- consistent with a small-ensemble (79 trees) model that leans on a different, less-generalizable set of secondary embedding features per seed, rather than a training-divergence or numerical-instability bug. No NaN/inf in either fit.


**Two separate conclusions, not one combined verdict:**
1. **LightGBM's best-model claim is confirmed and unchanged** -- its range doesn't overlap with RandomForest's or XGBoost's under either tree method.
2. **The original RandomForest-ahead-of-XGBoost ordering does not hold up** -- RandomForest's entire range sits inside XGBoost's range under both `hist` and `exact`. A real, separate finding: XGBoost's small tuned ensemble (79 trees) makes it seed-unstable on this target, in a way RandomForest and LightGBM are not.

## Part 4: PM100 Power

Fourth combination, in original completion order. `best_params` pulled from notebook 05 Part 2's committed output. Both `hist` and `exact` were run in full for XGBoost -- PM100's fits are cheap enough (2-3s each) that there's no cost trade-off to reason about, unlike F-DATA.

In [8]:
rf_p2 = joblib.load(f"{INTERIM_DIR}/pm100_power_seedvar_rf.joblib")
xgb_hist_p2 = joblib.load(f"{INTERIM_DIR}/pm100_power_seedvar_xgb.joblib")
xgb_exact_p2 = joblib.load(f"{INTERIM_DIR}/pm100_power_seedvar_xgb_exact.joblib")
lgbm_p2 = joblib.load(f"{INTERIM_DIR}/pm100_power_seedvar_lgbm.joblib")

power_p2_summary = pd.DataFrame({
    "RandomForest": summarize(rf_p2["seed_results"]),
    "XGBoost (hist)": summarize(xgb_hist_p2["seed_results"]),
    "XGBoost (exact)": summarize(xgb_exact_p2["seed_results"]),
    "LightGBM": summarize(lgbm_p2["seed_results"]),
}).T
print("PM100 power -- seed-variance summary, all variants:")
display(power_p2_summary.round(4))

print("\nPairwise range overlap (computed, not eyeballed):")
display(pairwise_overlaps(power_p2_summary))
print("\nhist and exact agree closely -- no F-DATA-memory-style instability here.")
print("RandomForest unambiguously behind both other models (no range overlap).")
print("XGBoost's range sits almost entirely inside LightGBM's wider range under either tree method --")
print("the original 'near tie' is confirmed as genuinely statistically indistinguishable, not a real ordering.")


PM100 power -- seed-variance summary, all variants:


,n_seeds,mean,std,min,max
RandomForest,5.0,0.8986,0.0050,0.8910,0.9030
XGBoost (hist),5.0,0.9230,0.0019,0.9207,0.9260
XGBoost (exact),5.0,0.9241,0.0039,0.9172,0.9291
LightGBM,5.0,0.9245,0.0068,0.9118,0.9311



Pairwise range overlap (computed, not eyeballed):


,overlap,region
pair,,
RandomForest vs XGBoost (hist),False,--
RandomForest vs XGBoost (exact),False,--
RandomForest vs LightGBM,False,--
XGBoost (hist) vs XGBoost (exact),True,"[0.9207, 0.9260]"
XGBoost (hist) vs LightGBM,True,"[0.9207, 0.9260]"
XGBoost (exact) vs LightGBM,True,"[0.9172, 0.9291]"



hist and exact agree closely -- no F-DATA-memory-style instability here.
RandomForest unambiguously behind both other models (no range overlap).
XGBoost's range sits almost entirely inside LightGBM's wider range under either tree method --
the original 'near tie' is confirmed as genuinely statistically indistinguishable, not a real ordering.


## Part 5: PM100 Memory (Clean-4)

Fifth combination, in original completion order. Feature set is the adopted clean-4 (`mem_req` deliberately excluded -- see the 2026-09-05 investigation). Both `hist` and `exact` run in full for XGBoost, same as PM100 power.

In [9]:
rf_m2 = joblib.load(f"{INTERIM_DIR}/pm100_memory_c4_seedvar_rf.joblib")
xgb_hist_m2 = joblib.load(f"{INTERIM_DIR}/pm100_memory_c4_seedvar_xgb_hist.joblib")
xgb_exact_m2 = joblib.load(f"{INTERIM_DIR}/pm100_memory_c4_seedvar_xgb_exact.joblib")
lgbm_m2 = joblib.load(f"{INTERIM_DIR}/pm100_memory_c4_seedvar_lgbm.joblib")

memory_p2_summary = pd.DataFrame({
    "RandomForest": summarize(rf_m2["seed_results"]),
    "LightGBM": summarize(lgbm_m2["seed_results"]),
    "XGBoost (hist)": summarize(xgb_hist_m2["seed_results"]),
    "XGBoost (exact)": summarize(xgb_exact_m2["seed_results"]),
}).T
print("PM100 memory (clean-4) -- seed-variance summary, all variants:")
display(memory_p2_summary.round(4))

print("\nPairwise range overlap (computed, not eyeballed):")
display(pairwise_overlaps(memory_p2_summary))

print("\nhist/exact agree closely -- no instability there. XGBoost's best-model claim (0.9658) "
      "survives cleanly against LightGBM (no overlap). But RandomForest vs LightGBM (2nd/3rd place) "
      "DOES overlap, in a small but real region -- that ordering is not reliably established, "
      "correcting an earlier characterization of this as a negligible boundary touch.")


PM100 memory (clean-4) -- seed-variance summary, all variants:


,n_seeds,mean,std,min,max
RandomForest,5.0,0.9509,0.0007,0.9500,0.9520
LightGBM,5.0,0.9582,0.0037,0.9519,0.9630
XGBoost (hist),5.0,0.9649,0.0010,0.9631,0.9658
XGBoost (exact),5.0,0.9644,0.0007,0.9631,0.9652



Pairwise range overlap (computed, not eyeballed):


,overlap,region
pair,,
RandomForest vs LightGBM,True,"[0.9519, 0.9520]"
RandomForest vs XGBoost (hist),False,--
RandomForest vs XGBoost (exact),False,--
LightGBM vs XGBoost (hist),False,--
LightGBM vs XGBoost (exact),False,--
XGBoost (hist) vs XGBoost (exact),True,"[0.9631, 0.9652]"



hist/exact agree closely -- no instability there. XGBoost's best-model claim (0.9658) survives cleanly against LightGBM (no overlap). But RandomForest vs LightGBM (2nd/3rd place) DOES overlap, in a small but real region -- that ordering is not reliably established, correcting an earlier characterization of this as a negligible boundary touch.


## Part 6: PM100 Duration

Sixth and final combination. Prepared data and `best_params` already existed from the earlier bounded-investigation work. Unlike PM100 memory/power, `exact` is *consistently* worse than `hist` here across all 5 seeds -- a real, directionally-stable difference, not an anomaly, confirming `hist` was the right call.

In [10]:
rf_d2 = joblib.load(f"{INTERIM_DIR}/pm100_duration_seedvar_rf.joblib")
xgb_hist_d2 = joblib.load(f"{INTERIM_DIR}/pm100_duration_seedvar_xgb_hist.joblib")
xgb_exact_d2 = joblib.load(f"{INTERIM_DIR}/pm100_duration_seedvar_xgb_exact.joblib")
lgbm_d2 = joblib.load(f"{INTERIM_DIR}/pm100_duration_seedvar_lgbm.joblib")

duration_p2_summary = pd.DataFrame({
    "RandomForest": summarize(rf_d2["seed_results"]),
    "XGBoost (hist)": summarize(xgb_hist_d2["seed_results"]),
    "XGBoost (exact)": summarize(xgb_exact_d2["seed_results"]),
    "LightGBM": summarize(lgbm_d2["seed_results"]),
}).T
print("PM100 duration -- seed-variance summary, all variants:")
display(duration_p2_summary.round(4))

print("\nPairwise range overlap (computed, not eyeballed):")
display(pairwise_overlaps(duration_p2_summary))
print("\nLightGBM's best-model claim confirmed, robust. RandomForest vs XGBoost 2nd/3rd place not "
      "reliably established -- XGBoost hist's range fully contains RandomForest's, same ambiguity "
      "already seen on F-DATA memory and PM100 power.")


PM100 duration -- seed-variance summary, all variants:


,n_seeds,mean,std,min,max
RandomForest,5.0,0.6074,0.0058,0.6011,0.6162
XGBoost (hist),5.0,0.6059,0.0133,0.5804,0.6185
XGBoost (exact),5.0,0.5445,0.0127,0.5248,0.5570
LightGBM,5.0,0.6968,0.0053,0.6881,0.7032



Pairwise range overlap (computed, not eyeballed):


,overlap,region
pair,,
RandomForest vs XGBoost (hist),True,"[0.6011, 0.6162]"
RandomForest vs XGBoost (exact),False,--
RandomForest vs LightGBM,False,--
XGBoost (hist) vs XGBoost (exact),False,--
XGBoost (hist) vs LightGBM,False,--
XGBoost (exact) vs LightGBM,False,--



LightGBM's best-model claim confirmed, robust. RandomForest vs XGBoost 2nd/3rd place not reliably established -- XGBoost hist's range fully contains RandomForest's, same ambiguity already seen on F-DATA memory and PM100 power.


## Summary: All Six Combinations Complete (corrected 2026-09-27)

Every "reliable?" cell below was re-checked with the computed `pairwise_overlaps` helper above, not eyeballed -- this corrects two cells from an earlier pass (F-DATA power's RF/LightGBM pair, PM100 memory's RF/LightGBM boundary case) that missed real overlaps.

| Target/Dataset | Best model | Survives? | 2nd/3rd reliable? |
|---|---|---|---|
| F-DATA power | XGBoost 0.9888 | Yes | **No** -- RF/LightGBM overlap `[0.9847, 0.9857]` |
| F-DATA duration | XGBoost 0.836 | Yes | Yes -- no overlaps anywhere |
| F-DATA memory | LightGBM 0.867 | Yes | No -- RF/XGBoost overlap (both tree methods) |
| PM100 power | XGBoost 0.9243 | No -- statistically tied with LightGBM | RF clearly behind both |
| PM100 memory | XGBoost 0.9658 | Yes | **No** -- RF/LightGBM overlap `[0.9519, 0.9520]`, small but real |
| PM100 duration | LightGBM 0.6994 | Yes | No -- RF/XGBoost(hist) overlap |

**Only F-DATA duration has every pairwise comparison genuinely non-overlapping.** Every other combination's runner-up ordering should be treated as statistically ambiguous, not as a secondary established fact, even where the top-of-ranking claim itself is robust.

Multi-seed variance testing (the first half of this work) is complete. Paired significance testing (notebook `08`, second half) is being redesigned around a user-clustered bootstrap of the R2 difference over test-set jobs, rather than seed-level Wilcoxon (which is underpowered at n=3-5 seeds by construction -- minimum achievable two-sided p is 0.0625 at n=5, 0.25 at n=3, neither can ever reach 0.05) -- design proposed separately before implementation.
